
# OBGPU Experiment Notebook

This notebook runs real `OBGPU` simulations through the stable benchmark path, then analyzes them using the same wavelet/LFP visualization standards as the older analysis notebooks by default.

Good starting points:
- `mode="fast"`, `nranks=1` for quick exploration
- `mode="parity"`, `nranks=2` when you care about matching the older reference behavior more closely
- `legacy_parallel_dt=False` when you want `sim_dt_ms` to control the runtime step more directly
- `legacy_parallel_dt=True` when you want the legacy parallel-dt behavior

All runs go to timestamped folders under `results/notebook_runs/`.


In [ ]:
import numpy as np
from pathlib import Path
import sys
from IPython.display import Image, display
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'obgpu_experiment_helpers.py').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import importlib
import obgpu_experiment_helpers as _obgpu_helpers
importlib.reload(_obgpu_helpers)
from obgpu_experiment_helpers import *

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
REPO_ROOT



## Available Controls

This prints the helper's supported top-level controls. You can still drop down to raw nested model overrides through `extra_overrides`.


In [ ]:

print_available_controls()



## Configure A Single Run

Edit this cell between runs. Most common experiment knobs are exposed directly here. For anything more specialized, use `extra_overrides`.

Examples of `extra_overrides`:
- `{"input_odors": {0: {"name": "Apple", "rel_conc": 0.1}, 200: {"name": "Mint", "rel_conc": 0.2}}}`
- `{"synapse_properties": {"AmpaNmdaSyn": {"ltpinvl": 0, "ltdinvl": 0}}}`
- `{"record_from_somas": ["MC", "TC"]}`


In [ ]:

RUN_CONFIG = build_run_config(
    mode='fast',              # 'fast' or 'parity'
    paramset='GammaSignature',
    label_prefix='experiment',
    nranks=1,                 # 1 is fastest here; 2 is the parity-oriented choice
    tstop_ms=1800.0,
    sim_dt_ms=0.1,
    recording_period_ms=0.1,
    legacy_parallel_dt=False,
    lfp_electrode_location=[116, 1078, -61],
    rnd_seed=0,
    mc_input_weight=0.2,
    tc_input_weight=0.8,
    mc_input_delay_ms=0.0,
    tc_input_delay_ms=0.0,
    gap_mc=32.0,
    gap_tc=32.0,
    ampa_nmda_gmax=64.0,
    ampa_nmda_nmdafactor=None,
    gaba_gmax=2.0,
    gaba_tau2_ms=36.0,
    max_firing_rate_hz=150.0,
    inhale_duration_ms=125.0,
    input_syn_tau1_ms=6.0,
    input_syn_tau2_ms=12.0,
    enable_lfp=True,
    record_from_somas=['MC', 'TC', 'GC'],
    record_gc_output_events=True,
    gc_output_bin_ms=5.0,
    gc_output_smooth_sigma_ms=10.0,
    gc_output_max_connections=120,
    spectrogram_signal='lfp',     # try 'mean_MC_voltage', 'mean_TC_voltage', or a soma label
    wavelet_signal='lfp',
    max_voltage_traces_per_type=4,
    max_spike_raster_cells_per_type=24,
    analysis_dt_ms=0.1,
    sniff_count=8,
    extra_overrides={
        # "input_odors": {0: {"name": "Apple", "rel_conc": 0.1}, 200: {"name": "Mint", "rel_conc": 0.2}},
        # "synapse_properties": {"GabaSyn": {"tau2": 50.0}},
    },
)

RUN_CONFIG = build_run_config(
#      mode="parity",
      paramset="GammaSignature",
      nranks=1,
      tstop_ms=1800.0,
      sim_dt_ms=0.1,
      recording_period_ms=0.1,
      #legacy_parallel_dt=True,
      enable_lfp=True,
        sniff_count=8,
      lfp_electrode_location=[  116, 1078, -61],
      record_from_somas=["MC", "TC", "GC"],
    extra_overrides = {"input_odors": {i: {"name": "Apple", "rel_conc": 0.5} for i in range(0,1800,200)}}
  )

RUN_CONFIG = build_run_config(
    mode="fast",
    nranks=1,
    paramset="GammaSignature",
    #ampa_nmda_nmdafactor=1000.0,
    kar_mt_gmax=2e-3,
    tstop_ms=1800.0*20,
    extra_overrides = {"input_odors": {i: {"name": "Apple", "rel_conc": 0.05} for i in range(0,1800,200)}},
    use_gpu=True,
    use_corenrn=True
)
RUN_CONFIG


## Optional: Run On Sol

Keep the notebook local and switch only the runner backend when you want Sol to do the compute. The analysis cells below stay the same because synced remote results are loaded into the normal `results/notebook_runs/...` layout.


In [ ]:
REMOTE_TARGET = "sol"  # "sol", "phx", or "local"

SOL_REMOTE_CONFIG = build_sol_remote_config(
    remote_host="jmpaniag@localhost",
    remote_repo_root="/scratch/jmpaniag/OlfactoryBulb",
    remote_results_root="/scratch/jmpaniag/OlfactoryBulb/results/notebook_runs",
    slurm_account="grp_scrook",   # e.g. "grp_scrook" if Sol requires it for your allocation
    remote_runtime_profiles=default_sol_runtime_profiles(),
    remote_heartbeat_timeout_s=60,
    slurm_time="03:30:00",
    slurm_gpus=1,
    slurm_cpus_per_task=None,
    slurm_partition="arm",
    slurm_reuse_allocation=True,
    ssh_options=['-p', '2222']
)

PHX_REMOTE_CONFIG = build_slurm_remote_config(
      remote_host="jmpaniag@localhost",
      remote_repo_root="/home/jmpaniag/OlfactoryBulb",
      remote_results_root="/home/jmpaniag/OlfactoryBulb/results/notebook_runs",
      remote_conda_activate_cmd="source tools/setup/activate_obgpu.sh",
      remote_fallback_conda_activate_cmd="source tools/setup/activate_obgpu.sh OBGPU-portable",
      remote_fast_node_feature="cascadelake",
      remote_fallback_mechanism_profile="portable",
      remote_mpi_exec="srun --mpi=pmix_v4 --cpu-bind=none",
      slurm_account="grp_scrook",
      remote_heartbeat_timeout_s=60,
      slurm_partition="general",
      slurm_time="3:30:00",
      slurm_gpus=None,
      slurm_cpus_per_task=1,
      ssh_options=["-p", "2223"],
      # Reusable allocations launch nested srun steps; keep disabled until the single-job path is clean.
      slurm_reuse_allocation=False,
      #slurm_extra_args=["--qos=private", "--ntasks=64", "--constraint=grp_scrook"],
      slurm_extra_args=["--qos=private", "--nodes=1", "--ntasks=16"],
  )

if REMOTE_TARGET == "sol":
    RUN_CONFIG.update(SOL_REMOTE_CONFIG)
elif REMOTE_TARGET == "phx":
    RUN_CONFIG.update(PHX_REMOTE_CONFIG)
    RUN_CONFIG["nranks"] = 16
    RUN_CONFIG["use_corenrn"] = False
    RUN_CONFIG["use_gpu"] = False
elif REMOTE_TARGET == "local":
    pass
else:
    raise ValueError(f"Unknown REMOTE_TARGET={REMOTE_TARGET!r}")

{
    "REMOTE_TARGET": REMOTE_TARGET,
    "runner_backend": RUN_CONFIG.get("runner_backend"),
    "remote_host": RUN_CONFIG.get("remote_host"),
    "remote_repo_root": RUN_CONFIG.get("remote_repo_root"),
    "slurm_partition": RUN_CONFIG.get("slurm_partition"),
    "slurm_reuse_allocation": RUN_CONFIG.get("slurm_reuse_allocation"),
    "remote_mpi_exec": RUN_CONFIG.get("remote_mpi_exec"),
    "remote_conda_activate_cmd": RUN_CONFIG.get("remote_conda_activate_cmd"),
    "remote_runtime_profiles": [profile.get("name") for profile in RUN_CONFIG.get("remote_runtime_profiles", [])],
    "remote_fallback_conda_activate_cmd": RUN_CONFIG.get("remote_fallback_conda_activate_cmd"),
    "remote_fast_node_feature": RUN_CONFIG.get("remote_fast_node_feature"),
    "remote_fallback_mechanism_profile": RUN_CONFIG.get("remote_fallback_mechanism_profile"),
    "remote_heartbeat_timeout_s": RUN_CONFIG.get("remote_heartbeat_timeout_s"),
    "remote_cleanup_stale_allocations": RUN_CONFIG.get("remote_cleanup_stale_allocations"),
    "nranks": RUN_CONFIG.get("nranks"),
    "use_gpu": RUN_CONFIG.get("use_gpu"),
    "use_corenrn": RUN_CONFIG.get("use_corenrn"),
}


In [ ]:
RUN_CONFIG


## Run The Simulation

Rerun this cell after changing `RUN_CONFIG`. Each run gets a fresh timestamped result folder.


In [ ]:

run, result = run_and_load(RUN_CONFIG)
print_run_summary(run, result, RUN_CONFIG)


In [ ]:
#result = load_result("/home/alek/OlfactoryBulb/results/notebook_runs/obgpu_experiment_GammaSignature_fast_20260328_171259")


## Legacy-Standard Plots

This uses the same main visualization standards as the older `LFP Wavelet Analysis` notebook: stacked MC/TC traces with input events, raw+band-passed LFP overlay, full wavelet map, and sniff-averaged wavelet power, included for easy comparison to the analysis done by Justas and Elli


In [ ]:

legacy = show_legacy_plots(
    result,
    sniff_count=RUN_CONFIG.get('sniff_count', 8),
    #dt=RUN_CONFIG['analysis_dt_ms'],
)


In [ ]:
import importlib
import obgpu_experiment_helpers as hlp
importlib.reload(hlp)
# %%
from scipy.stats import gaussian_kde
import obgpu_experiment_helpers as hlp
# Note: if seaborn is available in the OBGPU environment, it can be imported for even easier plotting.
# sns.kdeplot(all_t, all_f, bw_adjust=(KDE_BW_X, KDE_BW_Y)) would also work.

# --- Analysis Parameters ---
MODULUS = 1800
CELL_RANGE = range(0, 190)      # Range of cells to include in plots (e.g., range(len(result['soma_vs'])) for all)
CELL_TYPES = ['TC']               # List of cell types to include (e.g., ['MC', 'TC', 'GC']) or None for all
HEXBIN_GRIDSIZE_T = 60          # Hexbin grid density (Time)
HEXBIN_GRIDSIZE_F = 30          # Hexbin grid density (Frequency - smaller = more vertical smoothing)
NUM_TIME_BINS = 32              # Number of bins for the ridgeline plot
SHOW_HEXBIN = False             # Toggle 2D hexbin plot
SHOW_KDE1D = True              # Toggle 1D overall frequency KDE
SHOW_KDE2D = True               # Toggle 2D time-frequency KDE distribution
SHOW_RIDGELINE_KDE = False      # Toggle KDE distribution in ridgeline plot
SHOW_DOTS = False               # Toggle dot distribution (scatter)
SHOW_HIST2D = False             # Toggle 2D histogram distribution (using imshow)
STRIP_PLOT = True               # Align dots in columns (binned look)
DOT_SIZE = 5                    # Size of dots
DOT_ALPHA = 0.2                 # Transparency of dots
NUM_FREQ_BINS = 1              # Frequency bins for 2D histogram
KDE_RESOLUTION_T = 100 # Grid resolution for 2D KDE (Time)
KDE_RESOLUTION_F = 100          # Grid resolution for 2D KDE (Frequency)
KDE_BW_METHOD = 'scott'         # KDE bandwidth method ('scott', 'silverman', or a scalar - smaller = less smooth)
KDE_BW_X = .1                  # Time smoothing multiplier (smaller = more detail in time)
KDE_BW_Y = .25 # Frequency smoothing multiplier (larger = smoother in frequency)
KDE_F_RESOLUTION = 1600          # Number of points to sample the 1D KDE
MAX_F_HZ = 200                  # Maximum frequency to display in the plots
BIN_ALPHA = 0.5                 # Transparency of the time-binned KDEs
KDE_CMAP = 'inferno'            # Colormap for the 2D plots (e.g., 'inferno', 'viridis', 'magma', 'hot')
# --------------------------

# Collect data for the specified cells (CELL_RANGE) for population analysis
all_freq_t = []
all_freq = []

for i in CELL_RANGE:
    if i >= len(result['soma_vs']):
        break
    label, t, mp = result['soma_vs'][i]
    
    # Filter by cell type if specified
    if CELL_TYPES is not None:
        if not any(label.startswith(ctype) for ctype in CELL_TYPES):
            continue

    t_f, f = hlp.calculate_instantaneous_frequency(t, mp)
    if len(t_f) > 0:
        all_freq_t.append(t_f % MODULUS)
        all_freq.append(f)

all_t = np.concatenate(all_freq_t) if all_freq_t else np.array([])
all_f = np.concatenate(all_freq) if all_freq else np.array([])

# Original frequency calculation logic (Individual plot)
plt.figure(figsize=(12, 6))
plotted_count = 0
for i in CELL_RANGE:
    if i >= len(result['soma_vs']):
        break
    label, t, mp = result['soma_vs'][i]
    
    # Filter by cell type if specified
    if CELL_TYPES is not None:
        if not any(label.startswith(ctype) for ctype in CELL_TYPES):
            continue

    t = np.array(t)
    spikes = hlp.detect_spikes(t, mp)
    if len(spikes) > 1:
        t_freq = (spikes[:-1] + spikes[1:]) / 2 # use midpoints between spikes
        spiking_hz = 1000 / (np.diff(spikes))
        #plt.plot(t_freq, spiking_hz, alpha=0.3, label=f'Cell {label}' if plotted_count < 3 else None)
        plotted_count += 1

'''plt.xlabel('Time (ms)')
plt.ylabel('Frequency (Hz)')
plt.title('Individual Cell Spiking Frequencies')
plt.ylim(0, MAX_F_HZ)
plt.legend()
plt.show()
'''
# Now plot distributions/KDE as requested
if len(all_t) > 0:
    #tstop = result.get('tstop_ms') or RUN_CONFIG.get('tstop_ms', all_t.max() if len(all_t) > 0 else 1000)
    tstop = np.max(all_t)

    if SHOW_KDE1D:
        # KDE at specific time points could be complex, 
        # but we can show a 1D KDE of frequencies across all time
        plt.figure(figsize=(10, 5))
        # Overall frequency 1D KDE - use KDE_BW_Y for frequency smoothing
        kde = gaussian_kde(all_f, bw_method=KDE_BW_METHOD)
        # Adjust 1D frequency smoothing
        if KDE_BW_Y != 1.0:
            kde.covariance *= KDE_BW_Y**2
            kde.cho_cov = np.linalg.cholesky(kde.covariance)
            kde.log_det = 2*np.log(np.diag(kde.cho_cov * np.sqrt(2*np.pi))).sum()
        f_range = np.linspace(0, max(all_f)*1.1, KDE_F_RESOLUTION)
        plt.plot(f_range, kde(f_range))
        plt.fill_between(f_range, kde(f_range), alpha=0.3)
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Density')
        plt.title('Overall Frequency Distribution (1D KDE)')
        plt.xlim(0, MAX_F_HZ)
        hlp.save_figure(f"spike_based_{'_'.join(CELL_TYPES)}_KDE_1D_BWfreq{KDE_BW_Y}", run_or_result=result)
        plt.show()

    if SHOW_KDE2D:
        # 2D KDE estimation
        plt.figure(figsize=(14, 8))
        values = np.vstack([all_t, all_f])
        kernel = gaussian_kde(values, bw_method=KDE_BW_METHOD)
        
        # Adjust bandwidth for X and Y separately (anisotropic KDE)
        if KDE_BW_X != 1.0 or KDE_BW_Y != 1.0:
            # Adjust the covariance matrix elements
            # kernel.covariance is [ [Var_t, Cov_tf], [Cov_ft, Var_f] ]
            kernel.covariance[0, 0] *= KDE_BW_X**2
            kernel.covariance[1, 1] *= KDE_BW_Y**2
            # Adjust correlation term if present
            kernel.covariance[0, 1] *= KDE_BW_X * KDE_BW_Y
            kernel.covariance[1, 0] *= KDE_BW_X * KDE_BW_Y
            kernel.cho_cov = np.linalg.cholesky(kernel.covariance)
            kernel.log_det = 2*np.log(np.diag(kernel.cho_cov * np.sqrt(2*np.pi))).sum()

        # Create a grid for evaluation
        t_grid = np.linspace(0, tstop, KDE_RESOLUTION_T)
        f_grid = np.linspace(0, MAX_F_HZ, KDE_RESOLUTION_F)
        T, F = np.meshgrid(t_grid, f_grid)
        positions = np.vstack([T.ravel(), F.ravel()])
        Z = np.reshape(kernel(positions).T, T.shape)
        
        # Plot using imshow for a smooth look
        im = plt.imshow(Z, origin='lower', extent=[0, tstop, 0, MAX_F_HZ], 
                        aspect='auto', cmap=KDE_CMAP, interpolation='bilinear')
        for x in np.arange(0, tstop, 2000):
            plt.axvline(x, color='white', alpha=0.35)

        plt.colorbar(im, label='Density (KDE)')
        plt.xlabel('Time (ms)')
        plt.ylabel('Frequency (Hz)')
        plt.title('Estimated 2D Frequency Distribution (KDE)')
        hlp.save_figure(f"spike_based_{'_'.join(CELL_TYPES)}_KDE_2D_Mod{MODULUS}_BWt{KDE_BW_X}_BWfreq{KDE_BW_Y}", run_or_result=result)
        plt.show()

    # Time-binned distributions as requested
    t_bins = np.linspace(0, tstop, NUM_TIME_BINS + 1)
    bin_width = t_bins[1] - t_bins[0]

    if SHOW_DOTS or SHOW_RIDGELINE_KDE:
        plt.figure(figsize=(14, 8))
        # Draw vertical bin boundaries for a clearly "binned" look
        for tb in t_bins:
            plt.axvline(tb, color='gray', alpha=0.1, linestyle='--')

        for i in range(len(t_bins)-1):
            t_start, t_end = t_bins[i], t_bins[i+1]
            mask = (all_t >= t_start) & (all_t < t_end)
            if np.any(mask):
                bin_f = all_f[mask]
                
                if SHOW_DOTS:
                    if STRIP_PLOT:
                        # Align all dots in this bin to a single vertical line (the bin center)
                        # to make it look clearly "binned"
                        x_pos = np.full_like(bin_f, t_start + bin_width / 2)
                        plt.scatter(x_pos, bin_f, s=DOT_SIZE, alpha=DOT_ALPHA, color='black', edgecolors='none')
                    else:
                        # Add random jitter within the bin for a "dot distribution" look
                        jitter = np.random.uniform(0, bin_width * 0.8, size=len(bin_f))
                        plt.scatter(t_start + jitter, bin_f, s=DOT_SIZE, alpha=DOT_ALPHA, color='black', edgecolors='none')

                if SHOW_RIDGELINE_KDE and len(bin_f) > 2: # Need enough points for KDE
                    kde = gaussian_kde(bin_f, bw_method=KDE_BW_METHOD)
                    if KDE_BW_Y != 1.0:
                        kde.covariance *= KDE_BW_Y**2
                        kde.cho_cov = np.linalg.cholesky(kde.covariance)
                        kde.log_det = 2*np.log(np.diag(kde.cho_cov * np.sqrt(2*np.pi))).sum()
                    f_range = np.linspace(0, max(all_f)*1.1, KDE_F_RESOLUTION)
                    density = kde(f_range)
                    density /= density.max() # Normalize for plotting
                    
                    # Offset the KDE plot along the x-axis to create a ridgeline-like effect
                    plt.fill_betweenx(f_range, t_start, t_start + density * bin_width * 0.8, alpha=BIN_ALPHA)
                    plt.plot(t_start + density * bin_width * 0.8, f_range, linewidth=1, color='black', alpha=0.3)
        
        plt.xlabel('Time (ms)')
        plt.ylabel('Frequency (Hz)')
        plt.title('Frequency Distributions at Specific Time Bins')
        plt.ylim(0, MAX_F_HZ)
        plt.grid(True, alpha=0.3)
        plt.show()
else:
    print("No spikes detected.")
# %%
# Summary of shapes
if all_freq:
    print(f"Collected {len(all_freq)} cells for population analysis (filtered by {CELL_TYPES}).")
    print(f"Showing individual lines for {plotted_count} cells within range {CELL_RANGE}.")

# Direct GC inhibitory output from reciprocal GABA NetCon events
# This is a better readout of GC output than soma spikes alone.
hlp.plot_gc_output_overview(
    result,
    bin_ms=RUN_CONFIG.get('gc_output_bin_ms', 5.0),
    smooth_sigma_ms=RUN_CONFIG.get('gc_output_smooth_sigma_ms', 10.0),
    max_connections=RUN_CONFIG.get('gc_output_max_connections', 120),
)
hlp.save_figure('gc_inhibitory_output_overview', run_or_result=result)
plt.show()


In [ ]:
from scipy.stats import gaussian_kde
import numpy as np
import matplotlib.pyplot as plt
import obgpu_experiment_helpers as hlp

try:
    MODULUS
except NameError as exc:
    raise RuntimeError(
        "Run the soma spike analysis cell above first so the shared KDE parameters are defined."
    ) from exc

# GC inhibitory output analysis uses the same plotting controls as the soma spike analysis cell above.
GC_OUTPUT_TARGET_TYPES = ['MC', 'TC']   # Use ['MC'], ['TC'], or None for all reciprocal GABA outputs
GC_OUTPUT_RANGE = None                  # Use range(...) to inspect only a subset of GC->target connections

_gc_output_data = hlp.collect_gc_output_frequency_samples(
    result,
    indices=GC_OUTPUT_RANGE,
    target_types=GC_OUTPUT_TARGET_TYPES,
    modulus=MODULUS,
)
_gc_output_t = _gc_output_data['times']
_gc_output_f = _gc_output_data['freqs']
_gc_output_target_label = 'all' if not GC_OUTPUT_TARGET_TYPES else '_'.join(GC_OUTPUT_TARGET_TYPES)

if len(_gc_output_t) == 0:
    print(
        "No GC inhibitory output frequencies available. "
        "Rerun with record_gc_output_events=True or loosen GC_OUTPUT_TARGET_TYPES / GC_OUTPUT_RANGE."
    )
else:
    gc_output_tstop = np.max(_gc_output_t)
    print(
        f"Collected {_gc_output_data['n_events']} GC output connections with >=2 events "
        f"for target_types={GC_OUTPUT_TARGET_TYPES}."
    )

    if SHOW_KDE1D:
        plt.figure(figsize=(10, 5))
        kde = gaussian_kde(_gc_output_f, bw_method=KDE_BW_METHOD)
        if KDE_BW_Y != 1.0:
            kde.covariance *= KDE_BW_Y**2
            kde.cho_cov = np.linalg.cholesky(kde.covariance)
            kde.log_det = 2 * np.log(np.diag(kde.cho_cov * np.sqrt(2 * np.pi))).sum()
        f_upper = max(MAX_F_HZ, float(np.max(_gc_output_f)) * 1.1)
        f_range = np.linspace(0, f_upper, KDE_F_RESOLUTION)
        plt.plot(f_range, kde(f_range))
        plt.fill_between(f_range, kde(f_range), alpha=0.3)
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Density')
        plt.title(f'GC Inhibitory Output Frequency Distribution ({_gc_output_target_label})')
        plt.xlim(0, MAX_F_HZ)
        hlp.save_figure(
            f"gc_output_to_{_gc_output_target_label}_KDE_1D_BWfreq{KDE_BW_Y}",
            run_or_result=result,
        )
        plt.show()

    if SHOW_KDE2D and len(_gc_output_t) > 1:
        plt.figure(figsize=(14, 8))
        values = np.vstack([_gc_output_t, _gc_output_f])
        kernel = gaussian_kde(values, bw_method=KDE_BW_METHOD)

        if KDE_BW_X != 1.0 or KDE_BW_Y != 1.0:
            kernel.covariance[0, 0] *= KDE_BW_X**2
            kernel.covariance[1, 1] *= KDE_BW_Y**2
            kernel.covariance[0, 1] *= KDE_BW_X * KDE_BW_Y
            kernel.covariance[1, 0] *= KDE_BW_X * KDE_BW_Y
            kernel.cho_cov = np.linalg.cholesky(kernel.covariance)
            kernel.log_det = 2 * np.log(np.diag(kernel.cho_cov * np.sqrt(2 * np.pi))).sum()

        t_grid = np.linspace(0, gc_output_tstop, KDE_RESOLUTION_T)
        f_grid = np.linspace(0, MAX_F_HZ, KDE_RESOLUTION_F)
        T, F = np.meshgrid(t_grid, f_grid)
        positions = np.vstack([T.ravel(), F.ravel()])
        Z = np.reshape(kernel(positions).T, T.shape)

        im = plt.imshow(
            Z,
            origin='lower',
            extent=[0, gc_output_tstop, 0, MAX_F_HZ],
            aspect='auto',
            cmap=KDE_CMAP,
            interpolation='bilinear',
        )
        for x in np.arange(0, gc_output_tstop, 2000):
            plt.axvline(x, color='white', alpha=0.35)

        plt.colorbar(im, label='Density (KDE)')
        plt.xlabel('Time (ms)')
        plt.ylabel('Frequency (Hz)')
        plt.title(f'GC Inhibitory Output 2D KDE ({_gc_output_target_label})')
        hlp.save_figure(
            f"gc_output_to_{_gc_output_target_label}_KDE_2D_Mod{MODULUS}_BWt{KDE_BW_X}_BWfreq{KDE_BW_Y}",
            run_or_result=result,
        )
        plt.show()

    if SHOW_DOTS or SHOW_RIDGELINE_KDE:
        t_bins = np.linspace(0, gc_output_tstop, NUM_TIME_BINS + 1)
        bin_width = t_bins[1] - t_bins[0]

        plt.figure(figsize=(14, 8))
        for tb in t_bins:
            plt.axvline(tb, color='gray', alpha=0.1, linestyle='--')

        for i in range(len(t_bins) - 1):
            t_start, t_end = t_bins[i], t_bins[i + 1]
            mask = (_gc_output_t >= t_start) & (_gc_output_t < t_end)
            if np.any(mask):
                bin_f = _gc_output_f[mask]

                if SHOW_DOTS:
                    if STRIP_PLOT:
                        x_pos = np.full_like(bin_f, t_start + bin_width / 2)
                        plt.scatter(x_pos, bin_f, s=DOT_SIZE, alpha=DOT_ALPHA, color='black', edgecolors='none')
                    else:
                        jitter = np.random.uniform(0, bin_width * 0.8, size=len(bin_f))
                        plt.scatter(t_start + jitter, bin_f, s=DOT_SIZE, alpha=DOT_ALPHA, color='black', edgecolors='none')

                if SHOW_RIDGELINE_KDE and len(bin_f) > 2:
                    kde = gaussian_kde(bin_f, bw_method=KDE_BW_METHOD)
                    if KDE_BW_Y != 1.0:
                        kde.covariance *= KDE_BW_Y**2
                        kde.cho_cov = np.linalg.cholesky(kde.covariance)
                        kde.log_det = 2 * np.log(np.diag(kde.cho_cov * np.sqrt(2 * np.pi))).sum()
                    f_range = np.linspace(0, max(_gc_output_f) * 1.1, KDE_F_RESOLUTION)
                    density = kde(f_range)
                    density /= density.max()
                    plt.fill_betweenx(f_range, t_start, t_start + density * bin_width * 0.8, alpha=BIN_ALPHA)
                    plt.plot(t_start + density * bin_width * 0.8, f_range, linewidth=1, color='black', alpha=0.3)

        plt.xlabel('Time (ms)')
        plt.ylabel('Frequency (Hz)')
        plt.title(f'GC Inhibitory Output Distributions at Specific Time Bins ({_gc_output_target_label})')
        plt.ylim(0, MAX_F_HZ)
        plt.grid(True, alpha=0.3)
        hlp.save_figure(
            f"gc_output_to_{_gc_output_target_label}_time_binned_Mod{MODULUS}",
            run_or_result=result,
        )
        plt.show()


In [ ]:
# Optional one-time install if you want seaborn-specific plots later.
# %conda install seaborn


## Additional Helper Plots

These are extra views on top of the legacy-standard plots.


In [ ]:

plot_input_overview(
    result,
    bin_ms=RUN_CONFIG.get('input_bin_ms', 5.0),
    smooth_sigma_ms=RUN_CONFIG.get('input_smooth_sigma_ms', 10.0),
    max_segments=RUN_CONFIG.get('input_max_segments', 120),
    normalization=RUN_CONFIG.get('input_rate_normalization', 'per_target_cell'),
)
plt.show()

plot_named_signal(
    result,
    signal=RUN_CONFIG.get('spectrogram_signal', 'lfp'),
    dt_ms=RUN_CONFIG.get('analysis_dt_ms', 0.1),
)
plt.show()


In [ ]:

plot_voltage_traces(result, max_per_type=RUN_CONFIG['max_voltage_traces_per_type'])
plt.show()

plot_spike_raster(result, max_cells_per_type=RUN_CONFIG['max_spike_raster_cells_per_type'])
plt.show()

plot_gc_output_overview(
    result,
    bin_ms=RUN_CONFIG.get('gc_output_bin_ms', 5.0),
    smooth_sigma_ms=RUN_CONFIG.get('gc_output_smooth_sigma_ms', 10.0),
    max_connections=RUN_CONFIG.get('gc_output_max_connections', 120),
    normalization=RUN_CONFIG.get('gc_output_rate_normalization', 'per_target_cell'),
)
plt.show()


In [ ]:

plot_lfp_overview(result, dt_ms=RUN_CONFIG['analysis_dt_ms'])
plt.show()

plot_spectrogram(result, signal=RUN_CONFIG['spectrogram_signal'], dt_ms=RUN_CONFIG['analysis_dt_ms'], max_freq_hz=150.0)
plt.show()

plot_wavelet(result, signal=RUN_CONFIG['wavelet_signal'], dt_ms=RUN_CONFIG['analysis_dt_ms'])
plt.show()

plot_wavelet_band_power(result, signal=RUN_CONFIG['wavelet_signal'], dt_ms=RUN_CONFIG['analysis_dt_ms'])
plt.show()


In [ ]:

show_all_outputs(result, RUN_CONFIG)



## Single-Variable Sweep

Use a config key like `gaba_gmax`, `gap_mc`, `sim_dt_ms`, or a nested path like `lfp_electrode_location.2` or `extra_overrides.synapse_properties.GabaSyn.tau2`.


In [ ]:

SWEEP_CONFIG = dict(RUN_CONFIG)
RUN_OPTIONAL_SWEEP = False
SWEEP_CONFIG['label_prefix'] = 'sweep'
SWEEP_PATH = 'gaba_gmax'
SWEEP_VALUES = [0.0, 0.5, 1.0, 2.0, 4.0]

# Other good examples:
# SWEEP_PATH = 'gap_mc'
# SWEEP_PATH = 'sim_dt_ms'
# SWEEP_PATH = 'lfp_electrode_location.2'
# SWEEP_PATH = 'extra_overrides.synapse_properties.GabaSyn.tau2'

SWEEP_CONFIG, SWEEP_PATH, SWEEP_VALUES


In [ ]:

if RUN_OPTIONAL_SWEEP:
    sweep = run_parameter_sweep(SWEEP_CONFIG, SWEEP_PATH, SWEEP_VALUES)
    sweep_summary = [(item['value'], item['run'].result_dir.name) for item in sweep['items']]
else:
    sweep = {'items': []}
    sweep_summary = 'Skipping optional sweep. Set RUN_OPTIONAL_SWEEP = True to run it.'
sweep_summary



## Sweep Animations

These save GIFs to a timestamped animation folder and display them inline. The first three use the older visualization style more closely; the spectrogram animation is included as an extra.


In [ ]:

if RUN_OPTIONAL_SWEEP and sweep.get('items'):
    lfp_anim = animate_lfp_sweep(sweep, signal='lfp', dt_ms=RUN_CONFIG['analysis_dt_ms'], interval=1000)
    lfp_gif = save_animation(lfp_anim, f'{SWEEP_PATH}_legacy_lfp_sweep')
    display(Image(filename=str(lfp_gif)))
    lfp_gif
else:
    print('Skipping optional LFP sweep animation.')


In [ ]:

if RUN_OPTIONAL_SWEEP and sweep.get('items'):
    wavelet_anim = animate_wavelet_sweep(sweep, signal='lfp', dt_ms=RUN_CONFIG['analysis_dt_ms'], interval=1000)
    wavelet_gif = save_animation(wavelet_anim, f'{SWEEP_PATH}_legacy_wavelet_sweep')
    display(Image(filename=str(wavelet_gif)))
    wavelet_gif
else:
    print('Skipping optional wavelet sweep animation.')


In [ ]:

if RUN_OPTIONAL_SWEEP and sweep.get('items'):
    sniff_avg_anim = animate_sniff_average_sweep(sweep, dt_ms=RUN_CONFIG['analysis_dt_ms'], sniff_count=RUN_CONFIG.get('sniff_count', 8), interval=1000)
    sniff_avg_gif = save_animation(sniff_avg_anim, f'{SWEEP_PATH}_legacy_sniff_average_sweep')
    display(Image(filename=str(sniff_avg_gif)))
    sniff_avg_gif
else:
    print('Skipping optional sniff-average sweep animation.')


In [ ]:

if RUN_OPTIONAL_SWEEP and sweep.get('items'):
    spec_anim = animate_spectrogram_sweep(sweep, signal=RUN_CONFIG['spectrogram_signal'], dt_ms=RUN_CONFIG['analysis_dt_ms'], max_freq_hz=150.0, interval=1000)
    spec_gif = save_animation(spec_anim, f'{SWEEP_PATH}_spectrogram_sweep')
    display(Image(filename=str(spec_gif)))
    spec_gif
else:
    print('Skipping optional spectrogram sweep animation.')



## Re-open Older Notebook Runs


In [ ]:

recent_runs = list_notebook_runs(prefix=RUN_CONFIG['label_prefix'])
[path.name for path in recent_runs[-10:]]


In [ ]:
# Example:
# older_run, older_result = load_run_pair(recent_runs[-2])
# newer_run, newer_result = load_run_pair(recent_runs[-1])
# result_overview(older_result)
# legacy_old = load_legacy_wavelet_analysis(older_result, dt=RUN_CONFIG['analysis_dt_ms'], sniff_count=RUN_CONFIG.get('sniff_count', 8))
